In [5]:
from pathlib import Path
import csv
from collections import Counter

In [6]:
bbox_csv_path = Path('../data/raw/RAW_NIH/BBox_List_2017.csv')

with bbox_csv_path.open(newline='', encoding='utf-8') as file:
    reader = csv.DictReader(file)
    abnormality_counts = Counter(row['Finding Label'] for row in reader)

print(f'Total annotated abnormalities: {sum(abnormality_counts.values()):,}')
print('\nAbnormalities per classification:')
for finding_label, count in abnormality_counts.most_common():
    print(f'{finding_label}: {count:,}')


Total annotated abnormalities: 984

Abnormalities per classification:
Atelectasis: 180
Effusion: 153
Cardiomegaly: 146
Infiltrate: 123
Pneumonia: 120
Pneumothorax: 98
Mass: 85
Nodule: 79


In [ ]:
# Keep only these NIH findings, and rename some labels to match your target taxonomy
rename_map = {
    "Effusion": "Pleural effusion",
    "Infiltrate": "Infiltration",
}

keep_labels = {
    "Atelectasis",
    "Cardiomegaly",
    "Pneumothorax",
    "Effusion",
    "Infiltrate",
}

nih_bbox_csv_path = Path("../data/raw/RAW_NIH/BBox_List_2017.csv")
nih_out_csv_path = Path("../data/processed/01_nih_bbox_selected_renamed.csv")
nih_out_csv_path.parent.mkdir(parents=True, exist_ok=True)

total_rows = 0
kept_rows = 0

with nih_bbox_csv_path.open(newline="", encoding="utf-8") as infile, nih_out_csv_path.open(
    "w", newline="", encoding="utf-8"
) as outfile:
    reader = csv.DictReader(infile)

    out_fieldnames = list(reader.fieldnames or [])
    writer = csv.DictWriter(outfile, fieldnames=out_fieldnames)
    writer.writeheader()

    for row in reader:
        total_rows += 1
        label = (row.get("Finding Label") or "").strip()

        # Excludes Mass, Nodule, Pneumonia, etc. by only keeping `keep_labels`
        if label not in keep_labels:
            continue

        row["Finding Label"] = rename_map.get(label, label)  # Effusion->Pleural effusion, Infiltrate->Infiltration
        writer.writerow(row)
        kept_rows += 1

print(f"Read rows: {total_rows:,}")
print(f"Wrote rows (kept+renamed): {kept_rows:,}")
print(f"Output: {nih_out_csv_path.resolve()}")


Read rows: 984
Wrote rows (kept+renamed): 700
Output: D:\Home\Documents\GitHub\cxraide-data-pipeline\notebooks\data\processed\nih_bbox_selected_renamed.csv


In [4]:
# Input files
selected_csv_path = Path("../data/processed/01_nih_bbox_selected_renamed.csv")
data_entry_csv_path = Path("../data/raw/RAW_NIH/Data_Entry_2017.csv")

# Output file
output_csv_path = Path("../data/processed/02_nih_train_selected_classes_pa_only.csv")
output_csv_path.parent.mkdir(parents=True, exist_ok=True)

# Step 1: collect NIH image filenames that have View Position == PA
pa_image_names = set()

with data_entry_csv_path.open(newline="", encoding="utf-8") as file:
    reader = csv.DictReader(file)

    for row in reader:
        image_name = (row.get("Image Index") or "").strip()
        view_position = (row.get("View Position") or "").strip()

        if view_position == "PA":
            pa_image_names.add(image_name)

print(f"PA images found in Data_Entry_2017.csv: {len(pa_image_names):,}")

# Step 2: filter selected CSV using image_id/Image Index
total_rows = 0
kept_rows = 0

with selected_csv_path.open(newline="", encoding="utf-8") as infile, output_csv_path.open(
    "w", newline="", encoding="utf-8"
) as outfile:
    reader = csv.DictReader(infile)
    fieldnames = list(reader.fieldnames or [])

    writer = csv.DictWriter(outfile, fieldnames=fieldnames)
    writer.writeheader()

    for row in reader:
        total_rows += 1

        # Works whether your selected CSV uses NIH-style "Image Index" or VinBig-style "image_id"
        image_name = (
            row.get("Image Index")
            or row.get("image_id")
            or row.get("image")
            or ""
        ).strip()

        if image_name in pa_image_names:
            writer.writerow(row)
            kept_rows += 1

print(f"Read rows: {total_rows:,}")
print(f"Wrote PA-only rows: {kept_rows:,}")
print(f"Output: {output_csv_path.resolve()}")

PA images found in Data_Entry_2017.csv: 67,310
Read rows: 700
Wrote PA-only rows: 364
Output: D:\Home\Documents\GitHub\cxraide-data-pipeline\notebooks\data\processed\02_nih_train_selected_classes_pa_only.csv


In [7]:
bbox_csv_path = Path('../data/processed/02_nih_train_selected_classes_pa_only.csv')

with bbox_csv_path.open(newline='', encoding='utf-8') as file:
    reader = csv.DictReader(file)
    abnormality_counts = Counter(row['Finding Label'] for row in reader)

print(f'Total annotated abnormalities: {sum(abnormality_counts.values()):,}')
print('\nAbnormalities per classification:')
for finding_label, count in abnormality_counts.most_common():
    print(f'{finding_label}: {count:,}')


Total annotated abnormalities: 364

Abnormalities per classification:
Atelectasis: 100
Pleural effusion: 81
Cardiomegaly: 78
Pneumothorax: 63
Infiltration: 42
